In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
from transformers.models.t5.modeling_t5 import T5Attention , T5LayerNorm
import torch.nn as nn
import math

model = AutoModelForSeq2SeqLM.from_pretrained("VietAI/vit5-base", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("VietAI/vit5-base")

tokenizer

AttributeError: T5TokenizerFast has no attribute config

In [3]:
model.config

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 3072,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "torch_dtype": "float32",
  "transformers_version": "4.53.3",
  "use_cache": true,
  "vocab_size": 36096
}

In [4]:
model.encoder

T5Stack(
  (embed_tokens): Embedding(36096, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerFF(
          (DenseReluDense): T5DenseActDense(
            (wi): Linear(in_features=768, out_features=3072, bias=False)
            (wo): Linear(in_features=3072, out_features=768, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
            (act): ReLU()
          )
          (layer_norm): T5LayerNorm()
          (d

In [5]:
model.decoder

T5Stack(
  (embed_tokens): Embedding(36096, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerCrossAttention(
          (EncDecAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=F

In [6]:
total_params = sum(p.numel() for p in model.parameters())

trained_params = sum(p.numel() for p in model.parameters() if p.requires_grad == True)

print(f"total_params: {total_params}, trained_params: {trained_params}")

total_params: 225950976, trained_params: 225950976


In [7]:
sen = "My phone is displayed on the this table"
print(tokenizer.tokenize(sen))

['▁My', '▁ph', 'one', '▁is', '▁display', 'ed', '▁on', '▁the', '▁this', '▁ta', 'ble']


In [8]:
sen2 = "Hoặc đơn giản dùng luôn"
print(tokenizer.tokenize(sen2))

ids = tokenizer.encode(sen2)
tokenizer.decode(ids)

['▁Hoặc', '▁đơn', '▁giản', '▁dùng', '▁luôn']


'Hoặc đơn giản dùng luôn</s>'

In [ ]:
from typing import Optional, Union
class MultiQuery_attention(T5Attention):
    def __init__(self, config, has_relative_attention_bias=False,
        layer_idx: Optional[int] = None):
        super().__init__(config, has_relative_attention_bias,layer_idx)
        self.head = config.num_heads
        self.d_model = config.d_model
        self.d_h = self.d_model // self.head
        self.q = nn.Linear(self.d_model, self.d_model, bias= False) 
        self.k = nn.Linear(self.d_model, self.d_h,bias= False) 
        self.v = nn.Linear(self.d_model, self.d_h, bias= False)
        self.o = nn.Linear(self.d_model, self.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)



    def forward(
        self,
        hidden_states,
        mask=None,
        key_value_states=None,
        position_bias=None,
        past_key_value=None,
        layer_head_mask=None,
        query_length=None,
        use_cache=False,
        output_attentions=False,
        cache_position=None,
    ):
        batch_size, seq_len = hidden_states.shape[:2]

        q = self.q(hidden_states)  # [B, T, d_model]

        key_value_encoder = hidden_states if key_value_states is None else key_value_states
        k = self.k(key_value_encoder)  # [B, T, d_h]
        v = self.v(key_value_encoder)  # [B, T, d_h]

        q = q.view(batch_size, seq_len, self.head, self.d_h).transpose(1, 2)  # [B, head, T, d_h]
        k = k.unsqueeze(1)  # [B, 1, T, d_h]
        v = v.unsqueeze(1)  # [B, 1, T, d_h]

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_h)

        # Position bias
        if position_bias is None and self.has_relative_attention_bias:
            context_position = torch.arange(seq_len, dtype=torch.long, device=hidden_states.device)[:, None]
            memory_position = torch.arange(seq_len, dtype=torch.long, device=hidden_states.device)[None, :]
            relative_position = memory_position - context_position
            relative_position_bucket = self._relative_position_bucket(relative_position)
            position_bias = self.relative_attention_bias(relative_position_bucket)
            position_bias = position_bias.permute(2, 0, 1).unsqueeze(0)  # [1, head, T, T]

        if position_bias is not None:
            scores = scores + position_bias

        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(1).unsqueeze(2)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = nn.functional.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v)  # [B, head, T, d_h]

        attn_output = attn_output.transpose(1, 2).reshape(batch_size, seq_len, self.d_model)
        attn_output = self.o(attn_output)
        attn_output = self.dropout(attn_output)

        utputs = (attn_output, position_bias)

        if output_attentions:
            outputs = outputs + (attn_weights,)
        return outputs
    
    
def convert_model(model):
    for block in model.encoder.block:
        block.layer[0].SelfAttention = MultiQuery_attention(config = model.config)


    for block in model.decoder.block:
        block.layer[0].SelfAttention = MultiQuery_attention(config = model.config)
        block.layer[1].EncDecAttention = MultiQuery_attention(config = model.config)
    
    return model 

customzied_model = convert_model(model)
customzied_model

T5ForConditionalGeneration(
  (shared): Embedding(36096, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(36096, 768)
    (block): ModuleList(
      (0-11): 12 x T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): MultiQuery_attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=64, bias=False)
              (v): Linear(in_features=768, out_features=64, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dr

In [10]:
total_redesigned_params = sum(p.numel() for p in customzied_model.parameters())

print(f"After redisignning - Total params: {total_redesigned_params}, comparing to the intial Model: {total_redesigned_params/total_params * 100}%")

After redisignning - Total params: 187021824, comparing to the intial Model: 82.77097417804471%


In [11]:
def freeze_params(customzied_model): 
    # Đóng băng tất cả tham số
    for param in model.parameters():
        param.requires_grad = False

    # Mở train phần SelfAttention encoder
    for block in model.encoder.block:
        for param in block.layer[0].SelfAttention.parameters():
            param.requires_grad = True

    # Mở train phần SelfAttention và EncDecAttention decoder
    for block in model.decoder.block:
        for param in block.layer[0].SelfAttention.parameters():
            param.requires_grad = True
        for param in block.layer[1].EncDecAttention.parameters():
            param.requires_grad = True
    return model 

freezed_model = freeze_params(customzied_model) 

In [12]:
trained_params = sum(p.numel() for p in freezed_model.parameters() if p.requires_grad == True)

print(f"After freezing - Total params: {total_redesigned_params}, Trainable params: {round(trained_params/total_redesigned_params * 100, 5)}%")

print(f"After freezing - Trainable params: {round(trained_params/total_params * 100, 5)}%")

After freezing - Total params: 187021824, Trainable params: 24.59941%
After freezing - Trainable params: 20.36117%
